# Modelo eleitoral presidencial 2026

Notebook espelhado do relatorio Quarto. Use para explorar dados, testar variaveis e comparar modelos.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..')
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

def read_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

polls = read_csv(PROCESSED / 'pesquisas_validos.csv')
raw_polls = read_csv(RAW / 'pesquisas.csv')
elections = read_csv(RAW / 'eleicoes.csv')
forecasts = read_csv(PROCESSED / 'previsoes.csv')


## Base carregada

In [ ]:
print('Pesquisas processadas:', len(polls))
print('Pesquisas brutas:', len(raw_polls))
print('Eleicoes cadastradas:', len(elections))
print('Previsoes salvas:', len(forecasts))

## Variaveis candidatas

- Pesquisas: voto valido estimado, instituto, metodo, amostra, recencia.
- Tempo: dias do fim do campo ate a eleicao, dias da publicacao ate a eleicao.
- Ordem: sequencia da pesquisa no instituto/cenario e na campanha.
- Fundamentos: incumbencia, sucessor governista, aprovacao, economia, rejeicao.

In [ ]:
if polls.empty:
    print('Ainda nao ha pesquisas processadas. Preencha data/raw/pesquisas.csv e rode scripts/preparar_dados.py.')
else:
    display(polls.head())

## Cobertura por instituto

In [ ]:
data = polls if not polls.empty else raw_polls
if data.empty or 'instituto' not in data:
    print('Base de pesquisas ainda vazia.')
else:
    counts = data.drop_duplicates(['ano_eleicao', 'instituto', 'data_publicacao', 'cenario']).groupby('instituto').size().sort_values()
    ax = counts.plot(kind='barh', figsize=(9, 5), title='Cobertura por instituto')
    ax.set_xlabel('Pesquisas')
    ax.set_ylabel('Instituto')
    plt.show()

## Evolucao por candidato

In [ ]:
if polls.empty or 'voto_valido_estimado' not in polls:
    print('Sem serie temporal ainda.')
else:
    tmp = polls.copy()
    tmp['data_publicacao'] = pd.to_datetime(tmp['data_publicacao'])
    tmp['voto_valido_estimado'] = pd.to_numeric(tmp['voto_valido_estimado'])
    latest_year = tmp['ano_eleicao'].max()
    tmp = tmp[tmp['ano_eleicao'] == latest_year]
    fig, ax = plt.subplots(figsize=(9, 5))
    for candidate, group in tmp.groupby('candidato'):
        series = group.groupby('data_publicacao')['voto_valido_estimado'].mean().sort_index()
        ax.plot(series.index, series.values, marker='o', label=candidate)
    ax.set_title(f'Evolucao das pesquisas em {latest_year}')
    ax.set_ylabel('Votos validos estimados (%)')
    ax.legend()
    plt.show()